# Lab 24: Causal ML — Double Machine Learning
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 62 min Core + 20 min Write it yourself + 10 min Extension

---

**Learning Objectives:**
- Show on simulated data why LASSO on Y ~ D + X misreports a treatment effect, and why Double ML with the same LASSO does not
- Set up a DoubleML data object with the right outcome, treatment and covariate columns, leaving out bad controls
- Fit a Partially Linear Regression (PLR) model with random forest nuisance models and 5-fold cross-fitting
- Read the Average Treatment Effect (ATE) and its confidence interval, and rebuild the estimate from the residuals by hand
- Recognize a bad control (a column that is, or contains, the outcome, or a post-treatment mediator) and measure what including one does to the estimate
- Estimate Conditional Average Treatment Effects (CATE) by income quartile
- Finish a cross-fitted partialling-out estimator yourself
- (Extension) Ask how strong a hidden confounder would have to be to explain the estimate away

**Dataset:** 401(k) eligibility and household wealth: 9,915 households from the 1991 Survey of Income and Program Participation (Chernozhukov & Hansen), loaded with `doubleml.datasets.fetch_401K`

**Prerequisites:** Lab 13 (Frisch-Waugh-Lovell: partialling out by hand), Lab 10 (bad controls), Lab 15 (K-fold cross-validation), Lab 16 (LASSO) and Lab 19 (random forests).

**How this lab works:** Parts 1, 3, 4 and 5 are GUIDED: run each cell as it is and read the output. Part 2 has three blanks (`____`) for you to fill. Part 6 is three written answers. "Write it yourself" is required: you finish a short function whose first half is given. The Extension is optional.

**Plan on about 1¾ hours** for the required work: setup (about 12 minutes, with the install), Parts 1-6, Write it yourself, and the README and submission (about 12 minutes). A Double ML fit on the 401(k) data takes about 10 seconds, and can take 20-40 seconds on Colab: Steps 2 and 4 fit one each, Step 5 fits four, and the check after Write it yourself fits one more.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Three blanks (`____`)** in Step 2: the outcome column, the treatment column, and the list of columns that must not be controls. Each is a name or a short list of names.
> - **Write it yourself (required),** after Part 6: finish `dml_by_hand()`, a function that does Double ML's cross-fitting itself. The half for the outcome is given; you write the two lines for the treatment and the one-line estimate (three blanks). The check cell below it must print "Passed".
> - **Three written answers** in Part 6, two or three sentences each.
>
> **You run and read:** everything else. Setup and Parts 1, 3, 4 and 5 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The CHECK cell after Write it yourself is different: you only run it and read its message; you do not need to follow its code.
>
> **The Extension** (optional) runs as it is, with one question. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Colab does not come with `doubleml`, so the first line installs it. It is pinned to version 0.11.4, the
version this notebook's expected outputs come from: a later version may draw its folds differently and
print slightly different numbers. The same line upgrades scikit-learn to 1.8 or later, because `LassoCV`
changed in 1.8 and Part 1's numbers come from the newer version. The first two lines of output print your
Python and scikit-learn versions. If scikit-learn shows a version below 1.8, use **Runtime → Restart
session** and run this cell again.

`fetch_401K(return_type='DataFrame')` downloads the data as a pandas table, from the GitHub page of the
authors of the Double ML paper. It is a 1991 survey, so it never changes.

In [ ]:
# GUIDED — run as-is
# Setup: install doubleml, import the libraries, and load the 401(k) data
!pip install doubleml==0.11.4 "scikit-learn>=1.8" -q

import sys
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')   # doubleml and scikit-learn print warnings about future versions on many fits

from doubleml import DoubleMLData, DoubleMLPLR
from doubleml.datasets import fetch_401K
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV

data = fetch_401K(return_type='DataFrame')

print(f'Python {sys.version.split()[0]}')
print(f'scikit-learn {sklearn.__version__} (Part 1\'s numbers assume 1.8 or later)')
print(f'Dataset shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
data.head()

**Expected output:** your Python version, scikit-learn 1.8 or later, `Dataset shape: (9915, 14)`, the 14 column names, and the first
five households. The columns this lab uses:

| column | what it is |
|---|---|
| `net_tfa` | net total financial assets, in dollars: the **outcome** |
| `e401` | 1 if the employer offers a 401(k) plan: eligibility, the **treatment** |
| `age`, `inc`, `educ`, `fsize` | age, income, years of education, family size |
| `marr`, `twoearn`, `db`, `pira`, `hown` | 1 if married, two earners, a defined-benefit pension, an IRA, owns a home |
| `nifa` | non-401(k) financial assets: one part of `net_tfa` |
| `tw` | total wealth, which includes `net_tfa` |
| `p401` | 1 if the household takes part in a 401(k) plan: decided after, and because of, eligibility |

The next cell summarizes six of them. `.describe()` gives each column's count, mean, standard deviation,
minimum, quartiles and maximum, and `.round(2)` shows two decimals.

In [ ]:
# GUIDED — run as-is
# Setup, continued: summary statistics
print('=== Summary Statistics ===')
print(data[['net_tfa', 'e401', 'inc', 'age', 'fsize', 'educ']].describe().round(2))

The balance check compares eligible households (`e401` = 1) with the rest. `data[data['e401'] == 1]`
keeps the rows where the condition is True. In the table, `{v:>10s}` right-aligns a string in 10
characters, and `{x:+10.2f}` prints a number with its sign and two decimals.

In [ ]:
# GUIDED — run as-is
# Setup, continued: eligible against non-eligible households
treated = data[data['e401'] == 1]
control = data[data['e401'] == 0]

print('=== Treatment Balance Check ===')
print(f'Treated (401k eligible):     {len(treated):,} ({len(treated)/len(data)*100:.1f}%)')
print(f'Control (not eligible):      {len(control):,} ({len(control)/len(data)*100:.1f}%)')
print()
print(f'{"Variable":>10s}  {"Treated":>10s}  {"Control":>10s}  {"Diff":>10s}')
for v in ['net_tfa', 'inc', 'age', 'fsize', 'educ', 'marr']:
    t_mean = treated[v].mean()
    c_mean = control[v].mean()
    print(f'{v:>10s}  {t_mean:10.2f}  {c_mean:10.2f}  {t_mean - c_mean:+10.2f}')

**Expected output:** 3,682 eligible households (37.1%) and 6,233 others (62.9%). Eligible households hold
$19,559 more in net financial assets, but they also earn $15,368 more a year and have 0.88 more years of
education: firms that offer 401(k) plans tend to employ better-paid workers. So part of the $19,559 is
income and education, not eligibility, and a raw comparison overstates the effect. Parts 2 and 3 take that
part out.

## Part 1: The Regularization Bias Problem (GUIDED — 12 min)

Why not run LASSO on Y ~ D + X and read off the coefficient on D? LASSO treats the treatment D like any
other column of X, so its penalty does two things at once:

1. It shrinks the coefficient on D toward zero.
2. It under-fits the confounders, because the same penalty pulls their coefficients toward zero too. The
   confounding they should have absorbed then leaks into the coefficient on D.

Here the second effect is the bigger one, and it points in the direction of the confounding, not toward
zero. The confounders below raise both the chance of treatment and the outcome, so the leak is upward:
the selection story from the balance table. Part 1 shows this on simulated data, where the true effect is
5.0 by construction.

**Step 1** simulates 1,000 people with 250 covariates. 60 of them are confounders: each raises both the
chance of treatment and the outcome. With n = 1,000 and p = 250 the penalty has to be large, so LASSO cannot
fit those 60 properly. `np.random.normal(0, 1, size=(n, p))` is an n-by-p table of standard normal draws,
`np.linspace(1.0, 0.3, k)` gives k evenly spaced numbers from 1.0 down to 0.3, and `X_sim @ beta_d`
multiplies each person's covariates by their coefficients and adds them up (a matrix product).
`np.random.binomial(1, propensity)` draws a 1 with each person's probability, and a 0 otherwise. `1_000` is
1000; the underscore only makes it easier to read.

In [ ]:
# GUIDED — run as-is
# Step 1: simulate 1,000 people, 250 covariates, and a true treatment effect of exactly 5.0
np.random.seed(42)
n = 1_000
p = 250          # covariates
k = 60           # how many of them are confounders
TRUE_ATE = 5.0

X_sim = np.random.normal(0, 1, size=(n, p))

# the same 60 covariates drive treatment AND outcome: that is confounding
strength = np.linspace(1.0, 0.3, k)
beta_y = np.zeros(p)
beta_y[:k] = 1.8 * strength     # effect on the outcome
beta_d = np.zeros(p)
beta_d[:k] = 1.4 * strength     # effect on treatment

propensity = 1 / (1 + np.exp(-(X_sim @ beta_d)))   # the logistic function turns any number into a probability
D_sim = np.random.binomial(1, propensity)
Y_sim = TRUE_ATE * D_sim + X_sim @ beta_y + np.random.normal(0, 2.5, n)

Step 1, continued runs LASSO on Y ~ D + X and reads off the coefficient on D. `np.column_stack` puts D in
column 0 and the 250 covariates after it. `LassoCV` chooses its penalty, `alpha`, by 5-fold
cross-validation (Lab 16). Like every scikit-learn model, it is fitted with `.fit(X, y)`: X first, the
reverse of `sm.OLS(y, X)`. After the fit, `.coef_` holds the coefficients and `.alpha_` the chosen penalty
(a trailing `_` marks something learned in the fit). `lasso.coef_ != 0` is True for each coefficient that
is not zero, and `np.sum` counts the Trues.

In [ ]:
# GUIDED — run as-is
# Step 1, continued: LASSO on Y ~ D + X, and its coefficient on D
X_with_D = np.column_stack([D_sim, X_sim])
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_with_D, Y_sim)
lasso_ate = lasso.coef_[0]      # column 0 is D

print('=== Naive LASSO Approach ===')
print(f'True ATE:      {TRUE_ATE:.2f}')
print(f'LASSO ATE:     {lasso_ate:.2f}')
print(f'Bias:          {lasso_ate - TRUE_ATE:+.2f}')
print(f'Bias (%):      {(lasso_ate - TRUE_ATE) / TRUE_ATE * 100:+.1f}%')
print()
print(f'Chosen penalty: alpha = {lasso.alpha_:.4f}')
print(f'Non-zero LASSO coefficients: {np.sum(lasso.coef_ != 0)} of {len(lasso.coef_)}'
      f'  (there are {k} true confounders)')

**Expected output:** a LASSO ATE of 6.12, a bias of +1.12 (+22.4%), `alpha = 0.0599`, and 134 non-zero
coefficients of 251. The bias is upward, as the start of Part 1 said it would be.

Step 1, continued runs Double ML on the same data, with the same LASSO. Only how LASSO is used changes. `pd.DataFrame(X_sim, columns=...)` turns the array into a table with columns named
`x0` to `x249`. Then:
- `DoubleMLData(table, y_col=..., d_cols=..., x_cols=...)` tells DoubleML which column is the outcome,
  which is the treatment, and which are the controls.
- `DoubleMLPLR(...)` is the Partially Linear Regression model, $Y = \theta D + g(X) + \text{noise}$.
  `ml_l` is the model that predicts Y from X, and `ml_m` the one that predicts D from X: the two
  **nuisance models**. `n_folds=5` is **cross-fitting**: the rows are split into 5 folds, and each row's
  predictions come from models fitted on the other 4 folds. `n_rep=1` draws the folds once.
- `.fit()` fits the 2 × 5 models; `.coef[0]` is then $\hat\theta$, and `.se[0]` its standard error.

DoubleMLPLR draws its folds at random when it is created, so `np.random.seed(42)` comes right before it:
the same folds, and the same numbers, on every run.

In [ ]:
# GUIDED — run as-is
# Step 1, continued: Double ML on the same data, with the same LASSO as both nuisance models
sim_x = [f'x{j}' for j in range(p)]
sim = pd.DataFrame(X_sim, columns=sim_x)
sim['d'] = D_sim
sim['y'] = Y_sim

np.random.seed(42)
dml_sim = DoubleMLPLR(
    DoubleMLData(sim, y_col='y', d_cols='d', x_cols=sim_x),
    ml_l=LassoCV(cv=5, random_state=42, max_iter=10000),    # predicts y from the x's
    ml_m=LassoCV(cv=5, random_state=42, max_iter=10000),    # predicts d from the x's
    n_folds=5, n_rep=1,
)
dml_sim.fit()

print('=== DML on the same data, same learners ===')
print(f'DML ATE:       {dml_sim.coef[0]:.2f}   (se {dml_sim.se[0]:.2f})')
print(f'Bias:          {dml_sim.coef[0] - TRUE_ATE:+.2f}'
      f'   ({(dml_sim.coef[0] - TRUE_ATE) / TRUE_ATE * 100:+.1f}%)')
print(f'Naive LASSO minus the truth, in DML standard errors: {abs(lasso_ate - TRUE_ATE) / dml_sim.se[0]:.1f}')

**Expected output:** a DML ATE of 4.99 (se 0.24), a bias of −0.01 (−0.2%), and a naive LASSO estimate
4.6 DML standard errors from the truth. Same data, same penalty, a different bias. Part 6 asks why.

Seed 42 is a good draw, not the typical one. Over 200 seeds of this same design (n = 1,000, p = 250,
truth 5.0), DML averages 5.23 with an SD of 0.28, and its 95% intervals cover 5.0 about 82% of the
time; naive LASSO averages 6.08 and is above 5 in every run. So DML is much closer to the truth and
the LASSO bias is systematic, but at this sample size DML is not centred on 5.0 either. More rows
and fewer nuisance parameters are what close that last gap.

## Part 2: Setting Up the DML Estimator (YOUR TASK — 10 min)

Step 2 tells DoubleML which column is the outcome, which is the treatment, and which are the controls,
on the real 401(k) data. Fill in three blanks:

- `y_col`: the outcome's column name, a string such as `'age'`;
- `d_cols`: the treatment's column name, also a string;
- `BAD_CONTROLS`: a list of the three columns that must **not** be controls. Use the table under Setup
  and Lab 10: a control has to be fixed before the treatment happens, and must not be, or contain, the
  outcome.

"Every column except y and d" is the tempting answer, and Part 4 measures what it costs. `x_cols` is then
built for you. `[c for c in data.columns if c not in ...]` is a list comprehension: it keeps each column
name `c` that is not in the list after `not in`. `[y_col, d_cols] + BAD_CONTROLS` joins two lists into one.

In [ ]:
# YOUR TASK — fill in the three blanks (____)
# Step 2: name the outcome, the treatment, and the columns that must not be controls
y_col = ____            # a string
d_cols = ____           # a string
BAD_CONTROLS = ____     # a list of three strings

x_cols = [c for c in data.columns if c not in [y_col, d_cols] + BAD_CONTROLS]

dml_data = DoubleMLData(data, y_col=y_col, d_cols=d_cols, x_cols=x_cols)

print(dml_data)
print(f'\nOutcome variable: {y_col}')
print(f'Treatment variable: {d_cols}')
print(f'Excluded as bad controls: {BAD_CONTROLS}')
print(f'Covariates ({len(x_cols)}): {x_cols}')

**Expected output:** DoubleML's summary of the data, then
`Covariates (9): ['age', 'inc', 'fsize', 'educ', 'db', 'marr', 'twoearn', 'pira', 'hown']`. All nine are
treated as fixed before eligibility. If you see 10 or 11 covariates, one of the three is missing from your list; an
error from `DoubleMLData` usually means a misspelt name.

Step 2, continued fits the model, and it is given. Both nuisance models are random forests (Lab 19) of 200
trees, each at most 5 levels deep, with `random_state=42` so that the forests are the same on every run.
`ml_m` predicts eligibility, a 0/1 column, with a regression forest: its prediction is the share of
eligible households among similar ones, a number between 0 and 1. `np.random.seed(42)` fixes the folds,
as in Step 1; without it, every rerun of this cell would draw new folds and print a slightly different ATE
in Part 3. The fit takes about 10 seconds, and up to a minute on Colab.

In [ ]:
# GUIDED — run as-is
# Step 2, continued: two random forests as the nuisance models, and the fit
ml_l = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)   # predicts the outcome from x_cols
ml_m = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)   # predicts the treatment from x_cols

np.random.seed(42)
dml_plr = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5, n_rep=1)
dml_plr.fit()
print('DML model fitted.')

## Part 3: Reading the ATE (GUIDED — 12 min)

Step 3 reads the estimate off the fitted model. `.coef[0]` is the estimated effect $\hat\theta$, `.se[0]`
its standard error, and `.t_stat[0]` and `.pval[0]` the t-statistic and p-value (`[0]` picks the first, and
only, treatment). `.confint(level=0.95)` gives back a one-row table with the two ends of the 95% interval:
`.iloc[0, 0]` is row 0, column 0, the lower end. `:,.0f` prints a number with thousands commas and no
decimals. (`dml_plr.summary` holds the same numbers as a table.)

In [ ]:
# GUIDED — run as-is
# Step 3: the ATE, its standard error, its 95% interval, and its test
ate = dml_plr.coef[0]
se = dml_plr.se[0]
ci = dml_plr.confint(level=0.95)

print('=== DML Results: Effect of 401(k) Eligibility on Net Financial Assets ===')
print(f'Average Treatment Effect (ATE): ${ate:,.0f}')
print(f'Standard Error:                 ${se:,.0f}')
print(f'95% Confidence Interval:        [${ci.iloc[0, 0]:,.0f}, ${ci.iloc[0, 1]:,.0f}]')
print(f't-statistic:                    {dml_plr.t_stat[0]:.2f}')
print(f'p-value:                        {dml_plr.pval[0]:.6f}')

**Expected output:** an ATE of $8,633, a standard error of $1,304, a 95% interval of [$6,077, $11,189],
t = 6.62 and p = 0.000000 (below 0.0000005). Being eligible for a 401(k) plan raises net financial assets
by about $8,633, holding fixed the nine covariates; the interval excludes zero, so the effect is
significant at the 5% level. It is less than half of the raw $19,559 gap in the balance table.

**Step 3, continued: the same estimate, by hand.** DoubleML keeps the predictions it made while
cross-fitting: for each household, a prediction of its `net_tfa` and of its `e401`, each from the forest
fitted on the four folds that household was not in. `dml_plr.predictions['ml_l']` is a 3-D array
(household, repetition, treatment); `[:, 0, 0]` keeps every household, the first (only) repetition and the
first (only) treatment. `.values` is the plain NumPy array of a column.

From there, the last step is Lab 13's Frisch-Waugh-Lovell step. Subtract each prediction to get the
residuals, the part of wealth and the part of eligibility that the covariates do not explain, and regress
one leftover on the other. With no intercept, the slope of `y_resid` on `d_resid` is

$$\hat\theta = \frac{\sum_i \tilde d_i\, \tilde y_i}{\sum_i \tilde d_i^{\,2}}$$

where $\tilde y_i$ and $\tilde d_i$ are household $i$'s two residuals. In Lab 13, OLS made the two
predictions; here random forests make them, and each household's predictions come from forests that never
saw it.

Residualizing both sides has one more advantage: it makes $\hat\theta$ insensitive to small errors in the two
predictions. A forest that predicts wealth or eligibility a little wrong moves the estimate very little.
This property is called **Neyman orthogonality**, and it is why Double ML can use flexible models that are
never exactly right.

In [ ]:
# GUIDED — run as-is
# Step 3, continued: rebuild the ATE from DoubleML's cross-fitted predictions
l_hat = dml_plr.predictions['ml_l'][:, 0, 0]    # predicted outcome, from forests that never saw the household
m_hat = dml_plr.predictions['ml_m'][:, 0, 0]    # predicted treatment, the same way

y_resid = data[y_col].values - l_hat     # wealth the covariates do not explain
d_resid = data[d_cols].values - m_hat    # eligibility the covariates do not explain

theta = np.sum(d_resid * y_resid) / np.sum(d_resid ** 2)
print(f'by hand:  ${theta:,.2f}')
print(f'DoubleML: ${ate:,.2f}')

**Expected output:** `by hand:  $8,633.21` and `DoubleML: $8,633.21`, equal to the cent. All of Double ML's
machine learning is in the two predictions; the estimate itself is one line of arithmetic.

## Part 4: Bad Controls — What "Everything Else" Costs (GUIDED — 8 min)

Step 4 fits the same model again, with "every column except y and d" as the controls: `naive_x` keeps
`nifa`, `tw` and `p401`. The forests, the number of folds and the seed are the same as in Step 2, so the
folds are the same too, and the three columns are the only difference. The fit takes about 13 seconds, and
up to a minute on Colab.

In [ ]:
# GUIDED — run as-is
# Step 4: the same model, with nifa, tw and p401 among the controls
naive_x = [c for c in data.columns if c not in [y_col, d_cols]]

np.random.seed(42)
dml_naive = DoubleMLPLR(
    DoubleMLData(data, y_col=y_col, d_cols=d_cols, x_cols=naive_x),
    ml_l=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    ml_m=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    n_folds=5, n_rep=1,
)
dml_naive.fit()

ate_naive = dml_naive.coef[0]
ci_naive = dml_naive.confint(level=0.95)

print('=== Bad-control covariate set (all 12 remaining columns) ===')
print(f'Covariates ({len(naive_x)}): {naive_x}')
print()
print(f'ATE:            ${ate_naive:,.0f}')
print(f'Standard Error: ${dml_naive.se[0]:,.0f}')
print(f'95% CI:         [${ci_naive.iloc[0, 0]:,.0f}, ${ci_naive.iloc[0, 1]:,.0f}]')
print(f'p-value:        {dml_naive.pval[0]:.4f}')

**Expected output:** 12 covariates, an ATE of $-835, a standard error of $483, a 95% interval of
[$-1,782, $111], and p = 0.0836. Same data, same forests, same folds as Part 3's $8,633: three columns are
the whole difference, two that are, or contain, the outcome, and one post-treatment mediator. Lab 10 called these
bad controls; this is what they cost.

## Part 5: Heterogeneity — CATE by Income Quartile (GUIDED — 8 min)

Does the effect of eligibility differ with income? Step 5 splits the households into four income groups of
equal size, and Step 5, continued fits the Part 2 model within each group: the **Conditional Average
Treatment Effect** (CATE) for that group.

`pd.qcut(data['inc'], q=4, labels=[...])` cuts income at its quartiles and labels each household Q1 (the
lowest quarter) to Q4 (the highest). The labels are kept in their own Series, `quartile`, rather than added
to `data` as a column, so rerunning Step 2 or Step 4 later still finds the same columns.
`data[quartile == q]` keeps the households in quartile `q`.

In [ ]:
# GUIDED — run as-is
# Step 5: each household's income quartile
quartile = pd.qcut(data['inc'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print('Income quartile ranges:')
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    subset = data[quartile == q]
    print(f'  {q}: ${subset["inc"].min():,.0f} — ${subset["inc"].max():,.0f} '
          f'(n={len(subset):,})')

**Expected output:** four ranges, from $-2,652 — $19,413 (n=2,481) for Q1 to $48,585 — $242,124 (n=2,479)
for Q4.

Step 5, continued fits the Part 2 model on each quartile's households, `subset`, in turn.
`np.random.seed(42)` comes before each model, as in Step 2. Each pass adds a
dictionary of results to `cate_results`; `pd.DataFrame(cate_results)` turns that list of dictionaries into
a table with one row per dictionary, and `for _, row in cate_df.iterrows()` walks through its rows (`_` is
the row label, which is not needed). The four fits take about 10 seconds, and up to a minute on Colab.

In [ ]:
# GUIDED — run as-is
# Step 5, continued: the Part 2 model, fitted within each income quartile
cate_results = []
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    subset = data[quartile == q]

    np.random.seed(42)
    dml_q = DoubleMLPLR(
        DoubleMLData(subset, y_col=y_col, d_cols=d_cols, x_cols=x_cols),
        ml_l=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
        ml_m=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
        n_folds=5, n_rep=1,
    )
    dml_q.fit()
    ci_q = dml_q.confint(level=0.95)
    cate_results.append({'quartile': q, 'ate': dml_q.coef[0], 'se': dml_q.se[0],
                         'ci_lower': ci_q.iloc[0, 0], 'ci_upper': ci_q.iloc[0, 1]})

cate_df = pd.DataFrame(cate_results)
print('=== CATE by Income Quartile ===')
for _, row in cate_df.iterrows():
    print(f'{row["quartile"]}: ATE = ${row["ate"]:>10,.0f}  '
          f'95% CI = [${row["ci_lower"]:>10,.0f}, ${row["ci_upper"]:>10,.0f}]')

**Expected output:** Q1 $4,213 [$2,084, $6,342]; Q2 $3,421 [$969, $5,873]; Q3 $6,991 [$3,400, $10,582];
Q4 $19,084 [$11,178, $26,989].

Step 5, continued draws the four estimates with their 95% intervals. `yerr=[below, above]` draws each
error bar from the estimate down to the lower end of its interval and up to the upper end; a wider bar is
a less certain estimate.

In [ ]:
# GUIDED — run as-is
# Step 5, continued: the CATE by quartile, with 95% intervals
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(cate_df['quartile'], cate_df['ate'], color='steelblue', capsize=8,
       yerr=[cate_df['ate'] - cate_df['ci_lower'], cate_df['ci_upper'] - cate_df['ate']])
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Income Quartile')
ax.set_ylabel('Estimated ATE ($)')
ax.set_title('Conditional ATE of 401(k) Eligibility by Income Quartile')
plt.tight_layout()
plt.show()

## Part 6: Interpretation Questions (YOUR TASK — 12 min)

Answer these questions in your own words (2-3 sentences each):

**Q1:** Look at your CATE table and chart. Rank the quartiles by estimated effect. Intervals that do not
overlap show a real difference; overlapping ones are inconclusive. Which differences does that leave? What
economic mechanism could explain the pattern you see? (Think about who has income to spare, and who
already has assets to move.)

**Q2:** The note above Step 1's Double ML cell says what each LASSO predicts. Explain why errors in
those predictions barely move $\hat\theta$ once both Y and D are residualized, but move it a lot when D sits
beside X in one LASSO.

**Q3:** Part 4's standard error is smaller than Part 3's, yet Part 4's estimate is the one to distrust.
Why? Name the bad control you think does the damage, and say why.

*Your answers:*

1.

2.

3.

---

## Write it yourself (required — 20 min)

This part is required. The check cell below it must print "Passed" before you submit.

**The task.** Step 3, continued took DoubleML's cross-fitted predictions and did the last step by hand. Now
do the cross-fitting as well. The cell below holds `dml_by_hand(df, y_col, d_col, x_cols, model, folds)`
with its first half written: for each `(train, test)` pair in `folds`, it fits `model` to predict the
column `y_col` of `df` from the columns `x_cols`, on the `train` rows only, and stores the residual (actual
minus predicted) of each `test` row in `y_resid`. Fill in its three blanks:

1. and 2. inside the loop: the same two steps for `d_col`, storing the residuals in `d_resid`;
3. after the loop: give back the estimate from Step 3, continued's formula, as one number.

- `folds` is a list of `(train, test)` pairs of row positions. `dml_plr.smpls[0]` is the list of the five
  pairs DoubleML drew in Step 2. Each row is in exactly one `test`, so after the loop every row has been
  predicted once, by a model that never saw it. Lab 15's `kfold_mse()` had the same loop.
- `df[x_cols].iloc[train]` is the rows at the positions in `train`, and `a[test] = values` fills the
  positions in `test` of the array `a`.
- `.fit()` starts again from scratch every time it is called, so the same `model` can be fitted to `d_col`
  after `y_col`; but it has to be fitted again before it predicts `d_col`.
- Use the function's arguments, not the notebook's `data`, `y_col`, `x_cols` or `dml_plr`: the check also
  calls it on another table, with other column names, another model and other folds.
- Give back the estimate itself, not rounded.
- To test it yourself: `dml_by_hand(data, y_col, d_cols, x_cols, RandomForestRegressor(n_estimators=200,
  max_depth=5, random_state=42), dml_plr.smpls[0])` refits Step 2's forests on Step 2's folds, so it should
  give Step 3's ATE, $8,633.21. It takes about 10 seconds. Python may display it as
  `np.float64(8633.21...)`: that is just a number.

Then run the check cell. It calls your function twice: on this lab's data, with Step 2's forests and folds
(about 10 seconds, up to half a minute on Colab), and on 6,000 of the households in a shuffled order, with
every column renamed, a linear regression as the model, and 4 folds. `assert` stops with its message when
its condition is False; the message tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): finish dml_by_hand(df, y_col, d_col, x_cols, model, folds)
def dml_by_hand(df, y_col, d_col, x_cols, model, folds):
    "Double ML's partialling-out estimate, with the two nuisance models cross-fitted on folds."
    y_resid = np.zeros(len(df))
    d_resid = np.zeros(len(df))
    for train, test in folds:
        model.fit(df[x_cols].iloc[train], df[y_col].iloc[train])
        y_resid[test] = df[y_col].iloc[test] - model.predict(df[x_cols].iloc[test])
        ____    # 1. fit model to d_col, on the train rows
        ____    # 2. store d_col's residuals for the test rows in d_resid
    return ____    # 3. Step 3, continued's formula

In [ ]:
# CHECK — for "Write it yourself" above
import copy
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

# globals() holds every name defined so far in this notebook
# co_names lists the names a function uses: a blank still in it means the function is not finished
if "dml_by_hand" not in globals() or "_" * 4 in dml_by_hand.__code__.co_names:
    print("Not written yet: fill in the three blanks in dml_by_hand() above and run that cell, then run this check.")
elif not all(name in globals() for name in ["dml_plr", "ate", "x_cols", "y_col", "d_cols", "data"]):
    print("Run Parts 1-5 first, then this check.")
else:
    NINE = ["age", "inc", "fsize", "educ", "db", "marr", "twoearn", "pira", "hown"]
    assert y_col == "net_tfa" and d_cols == "e401" and list(x_cols) == NINE, (
        "Step 2's columns are not the ones its expected output lists. Fix Step 2's blanks, rerun Parts 2-5, "
        "then this cell.")
    assert ate == dml_plr.coef[0], "ate is not Step 2's estimate any more. Rerun Parts 2-5, then this cell."

    # What some common slips give, recorded for this lab's data with Step 2's forests and folds
    # (MAIN), and for the second input below with a linear regression and 4 folds (OTHER).
    MAIN = {"in-sample": 8595.317914746603, "per-fold": 8601.931030570528, "swapped": 8335.3069642263,
            "one model": 8553.581047928754, "only y": 5466.0049418623, "only y, intercept": 8411.003358604197,
            "only d": 8553.578068503666, "only d, intercept": 8566.444291240323,
            "intercept": 8633.339329777882, "cov/var": 8634.210152788755,
            "return in loop": 6507.270679704233, "arrays in loop": 14032.75499269726,
            "no refit": -0.027571379509800754}
    MAIN_OWN_FOLDS = [8782.536793209008, 8938.670375101728]
    OTHER = {"right": 6532.67058886622, "in-sample": 6241.877122455223, "per-fold": 6525.435529168035,
             "swapped": 6468.790219974603, "one model": 6363.8934441147785, "only y": 3481.3004360666214,
             "only y, intercept": 5458.577343318433, "only d": 6363.8920758655595,
             "only d, intercept": 6368.4683789631945, "intercept": 6532.680517564307,
             "cov/var": 6533.7694791441645, "return in loop": 8357.244324901923,
             "arrays in loop": 9569.79589330293, "no refit": 0.01861769897056238}
    OTHER_OWN_FOLDS = [6237.156810431363, 6291.328839536296, 5903.459748483963]
    OTHER_FOREST = 9884.680947242672

    def call(df, y, d, xs, model, folds, where):
        """Run your dml_by_hand on copies, so it cannot change the lab's data or models; say where to look if it fails."""
        # try/except catches an error inside your function, so the message can name the cause
        try:
            out = dml_by_hand(df.copy(), y, d, list(xs), copy.deepcopy(model), folds)
        except Exception as e:
            msg = str(e)
            hint = "Read the error above: it says what went wrong."
            if (isinstance(e, TypeError) and ("positional argument" in msg or "unexpected keyword" in msg)) or (
                    "has no attribute 'fit'" in msg or "estimators_" in msg or "not iterable" in msg):
                hint = "dml_by_hand takes six arguments, in this order: df, y_col, d_col, x_cols, model, folds."
            elif isinstance(e, KeyError) and any(repr(name) in msg for name in [y, d] + list(xs)):
                hint = ("Your function looks this table's columns up in another table: it uses the notebook's "
                        "data instead of its df argument.")
            elif isinstance(e, KeyError) and any(repr(name) in msg for name in ["net_tfa", "e401"] + NINE):
                hint = ("Your function asks for a column this table does not have: a name typed in, or the "
                        "notebook's y_col, d_cols or x_cols. Use the arguments y_col, d_col and x_cols.")
            elif isinstance(e, KeyError):
                hint = ("folds holds row positions, but [] and .loc look rows up by their index label. Use "
                        ".iloc[...] to take rows by position.")
            elif isinstance(e, IndexError) or "out-of-bounds" in msg or "out of bounds" in msg:
                hint = ("A position is beyond the end of the table. Does your function use its folds "
                        "argument, or Step 2's dml_plr.smpls?")
            # msg[:200] keeps the first 200 characters of the error message
            raise AssertionError(f"dml_by_hand on a table of {len(df):,} rows stopped with an error:\n"
                                 f"  {type(e).__name__}: {msg[:200]}\n{hint}") from None
        assert out is not None, "dml_by_hand gave back nothing. Does it end with a return line?"
        assert np.size(out) != len(df), (
            f"dml_by_hand gives back {len(df):,} values, one per row: residuals, not the estimate. Give back "
            "the one number from Step 3, continued's formula.")
        assert np.size(out) == 1, (
            f"dml_by_hand gives back {np.size(out)} values. Give back one number: the estimate.")
        value = float(np.ravel(out)[0])
        assert not (np.isnan(value) and where.startswith("On this lab")), (
            f"{where}, dml_by_hand gives nan: zero divided by zero, because every d residual is zero. "
            "d_resid is probably an integer array, made from the 0/1 column, so each residual is cut to 0 "
            "when it is stored. Make the residual array float (astype(float), or np.zeros(len(df)) as given).")
        assert not np.isnan(value), (
            f"{where}, dml_by_hand gives nan: zero divided by zero, because every d residual is zero. Two "
            "usual causes: d_resid is an integer array (make it float, as np.zeros(len(df)) is), or the rows "
            "used are not this table's rows (use the df argument everywhere, and take rows by position).")
        return value

    def near(a, b, rtol=1e-7):
        return np.isclose(a, b, rtol=rtol, atol=0)

    def rounded(got, want):
        """True if got is want rounded to some number of decimals (a negative number rounds to tens, hundreds, ...)."""
        return got != want and any(got == round(want, places) for places in range(-3, 5))

    def diagnose(got, want, slips, own_folds, where):
        """Name the slip behind a wrong answer, if it is a common one."""
        assert not near(got, slips["no refit"], 1e-6), (
            f"{where}, your d residuals come from the model still fitted to y_col: it predicts wealth, not "
            "eligibility. Fit the model to d_col on the train rows before you predict d_col.")
        assert abs(got) > 1e-3, (
            f"{where}, your estimate is almost zero ({got:.2e}): you divided by the sum of the squared "
            "residuals of y. Step 3, continued divides by the squared residuals of d.")
        assert not near(got, -want), "Your estimate has the wrong sign. Subtract the prediction from the actual value."
        assert not near(got, slips["in-sample"], 1e-5), (
            f"{where}, your residuals come from models fitted on every row, the predicted rows included: "
            "there is no cross-fitting. Fit on the train rows only, and predict the test rows.")
        assert not near(got, slips["swapped"]), (
            f"{where}, your models are fitted on the test rows and predict the train rows. Each pair in "
            "folds is (train, test), in that order.")
        assert not near(got, slips["return in loop"]), (
            f"{where}, your function gives back its answer after the first fold: the return line is inside "
            "the for loop. Move it out, so that every fold runs first.")
        assert not near(got, slips["arrays in loop"]), (
            f"{where}, only the last fold's residuals survive: the residual arrays are made again inside the "
            "loop, which wipes out the earlier folds. Make them once, before the loop.")
        assert not near(got, slips["per-fold"]), (
            f"{where}, your answer is the average of one estimate per fold. Fill in every row's residuals "
            "first, then apply the formula once, to all rows together.")
        assert not near(got, slips["one model"], 1e-8), (
            f"{where}, both residuals use the model fitted to d_col: the model was fitted to y_col, then "
            "fitted again to d_col before its y prediction was made. Predict right after each fit.")
        assert not (near(got, slips["only y"]) or near(got, slips["only y, intercept"])), (
            f"{where}, your estimate uses the residuals of y but the raw d. Take d's residuals too: both "
            "sides are partialled out.")
        assert not (near(got, slips["only d"], 1e-8) or near(got, slips["only d, intercept"])), (
            f"{where}, your estimate uses the residuals of d but the raw y. Take y's residuals too: both "
            "sides are partialled out.")
        assert not near(got, slips["intercept"], 1e-8), (
            f"{where}, close but not equal: your last step fits an intercept, or centers the residuals "
            "first (LinearRegression() and sm.add_constant both add one). Step 3, continued's formula has "
            "no intercept: use it as it is.")
        assert not near(got, slips["cov/var"]), (
            f"{where}, close but not equal: np.cov divides by n - 1 and np.var by n, and the formula also "
            "centers the residuals. Use Step 3, continued's formula as it is.")
        assert not any(near(got, v, 1e-6) for v in own_folds), (
            f"{where}, your function makes folds of its own instead of using its folds argument. Loop "
            "over the pairs in folds.")

    # 1. This lab's data, Step 2's forests and folds: the answer is Step 3's ATE
    forest = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
    got = call(data, y_col, d_cols, x_cols, forest, dml_plr.smpls[0], "On this lab's data")
    assert not rounded(got, ate), "Your number is rounded. Give back the full number; round only when you print."
    if not near(got, ate, 1e-6):
        diagnose(got, ate, MAIN, MAIN_OWN_FOLDS, "On this lab's data")
        assert False, (
            f"On this lab's data, with Step 2's forests and folds, dml_by_hand gives {got:,.2f}; it should "
            f"give Step 3's ATE, {ate:,.2f}. Check three things: each model is fitted on the train rows and "
            "predicts the test rows, both y_col and d_col are partialled out, and the last step is Step 3, "
            "continued's formula.")

    # 2. 6,000 of the households in a shuffled order, every column renamed, a linear regression, 4 folds
    renamed = {y_col: "wealth", d_cols: "eligible", **{c: "x_" + c for c in x_cols}}
    other = data[[y_col, d_cols] + list(x_cols)].astype(float).sample(n=6000, random_state=3).rename(columns=renamed)
    other_x = [renamed[c] for c in x_cols]
    other_folds = list(KFold(n_splits=4, shuffle=True, random_state=1).split(other))
    got2 = call(other, "wealth", "eligible", other_x, LinearRegression(), other_folds, "On the second table")
    assert not rounded(got2, OTHER["right"]), (
        "Your number is rounded. Give back the full number; round only when you print.")
    if not near(got2, OTHER["right"], 1e-6):
        assert not near(got2, OTHER_FOREST, 1e-4), (
            "On the second table, your function fits a random forest, not the model it is given. Fit the "
            "model argument.")
        assert not near(got2, got, 1e-9), (
            "On the second table, dml_by_hand gives the same answer as on the lab's data. Does it use its "
            "df argument, or the notebook's data?")
        diagnose(got2, OTHER["right"], OTHER, OTHER_OWN_FOLDS, "On the second table")
        assert False, (
            f"On the second table dml_by_hand gives {got2:,.2f}; it should give {OTHER['right']:,.2f}. It "
            "passed on the lab's data, so look for what the second call changes: its df (rows in a shuffled "
            "order, with the index labels shuffled too), its column names, its model, and 4 folds instead of "
            "5. Use only the arguments, and take rows by position.")

    print("Passed. dml_by_hand reproduces Step 3's ATE from Step 2's folds, and uses the table, columns, "
          "model and folds it is given.")
    print(f"\n  dml_by_hand, this lab's data:  {got:,.2f}")
    print(f"  DoubleML, Step 3:              {ate:,.2f}")

---

## Extension (Optional — 10 min): How Strong Would a Hidden Confounder Have to Be?

Double ML removes the confounding that the nine covariates measure, and nothing else. Financial literacy
or patience is not in the data, and could raise both the chance of working for a firm with a 401(k) and a
household's savings. `sensitivity_analysis()` asks how strong such a confounder would have to be to change
the answer.

`cf_y=0.03` and `cf_d=0.03` describe a confounder that explains about 3% of the variation in wealth, and
about 3% of the variation in eligibility, that the nine covariates leave unexplained. Step 6 prints:
- **Bounds with CI:** the range the estimate could lie in with such a confounder (`theta lower` to
  `theta upper`), and the 95% interval widened to match (`CI lower` to `CI upper`);
- **Robustness Values:** `RV (%)` is how strong a confounder (in the same sense, on both sides at once)
  would have to be to pull the estimate to zero, and `RVa (%)` how strong for the 95% interval to reach
  zero.

**Expected output:** the estimate, $8,633, could lie anywhere from about $4,951 to $12,315 with such a
confounder (interval $2,755 to $14,500); `RV (%)` is about 6.9 and `RVa (%)` about 5.1.

**Your answer.** Name one thing the data leave out, and argue in two or three sentences whether it could
be as strong as `RV (%)` says it would have to be.

In [ ]:
# GUIDED — run as-is (optional)
# Step 6: how far a confounder of 3% strength on each side could move Part 3's estimate
dml_plr.sensitivity_analysis(cf_y=0.03, cf_d=0.03)
print(dml_plr.sensitivity_summary)

*Your answer here:*

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* On simulated data with a true treatment effect of 5.0, LASSO on
  Y ~ D + X estimated [YOUR VALUE]; Double ML with the same LASSO
  estimated [YOUR VALUE]
* Estimated the effect of 401(k) eligibility on net financial assets
  for 9,915 households (1991 SIPP) with Double ML, random forests and
  5-fold cross-fitting: [YOUR VALUE], 95% interval [YOUR VALUE]
* Added three bad controls (nifa, tw, p401) and got [YOUR VALUE]
* Estimated the effect within each income quartile: [YOUR VALUES]
* Wrote dml_by_hand(), my own cross-fitted partialling-out estimator,
  and matched the Double ML estimate: [YOUR VALUE]

**Please write a README.md entry including:**
1. Project Title: The Effect of 401(k) Eligibility on Savings, with Double Machine Learning
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab24-double-ml`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab24-double-ml`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab24-double-ml`, branch `main`,
   commit message `Lab 24 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.